# 07 — RAG query (retrieval + LLM)

Run **after** notebook `06` so the index is populated.
Asks a question, prints the streamed answer plus the sources used.

In [1]:
import sys, os, asyncio
sys.path.insert(0, os.path.abspath('..'))
from src.search_client import SearchService
from src.openai_client import OpenAIService
from src.cosmos_client import create_state_service
from src.rag import RAGEngine

# Returns CosmosService if configured, else LocalStateService (file-backed JSON).
state = create_state_service()
print('State backend:', type(state).__name__)
rag = RAGEngine(SearchService(), OpenAIService(), state)

INFO: Incomplete environment configuration for EnvironmentCredential. These variables are set: AZURE_TENANT_ID
INFO: ManagedIdentityCredential will use IMDS


State backend: LocalStateService


In [2]:
QUESTION = 'give me the flow diagram.'
answer, sources, turn = rag.answer(QUESTION, session_id='nb-rag-session', user_id='nb-user')
print(answer)
print('\n--- sources ---')
for s in sources:
    print(f'  {s.doc_id}  page={s.page}  type={s.type}  score={s.snippet[:60]}…')

INFO: HTTP Request: POST https://azr-oai-dai-sandbox4-001.openai.azure.com/openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-08-01-preview "HTTP/1.1 200 OK"
INFO: Request URL: 'https://agent-ai-search5yc2.search.windows.net/indexes('pdg-was-multimodal-rag-2')/docs/search.post.search?api-version=2024-05-01-preview'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '34637'
    'api-key': 'REDACTED'
    'Accept': 'application/json;odata.metadata=none'
    'x-ms-client-request-id': '5f41cd7f-4882-11f1-bc65-df1619e59047'
    'User-Agent': 'azsdk-python-search-documents/11.6.0b4 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
A body is sent with the request
INFO: Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; odata.metadata=none; odata.streaming=true; charset=utf-8'
    'Content-Encoding': 'REDACTED'
    'Vary': 'REDACTED'
    'Strict-Transport-Security': 'REDAC

See the flowchart image detailing the process for handling a user query through various nodes and tasks on page 15.

--- sources ---
  90ab578f-35a7-4314-9369-4263b6c47d53  page=14  type=text  score=6. Architecture Diagram
The following diagram illustrates th…
  90ab578f-35a7-4314-9369-4263b6c47d53  page=16  type=text  score=Figure 1: Multi-Agent Research System - Architecture Flow Di…
  90ab578f-35a7-4314-9369-4263b6c47d53  page=23  type=text  score=8. Control Flow & Feedback Loops
The system implements two c…
  90ab578f-35a7-4314-9369-4263b6c47d53  page=13  type=text  score=5. System Architecture Overview
The system follows a directe…
  90ab578f-35a7-4314-9369-4263b6c47d53  page=15  type=image  score=[Image on page 15]: This image is a flowchart detailing a pr…


In [3]:
# Streaming variant
async def run():
    async for evt in rag.stream_answer('What images appear in this doc?', 'nb-rag-session', 'nb-user'):
        if evt['type'] == 'token':
            print(evt['content'], end='', flush=True)
        elif evt['type'] == 'sources':
            print(f"[{len(evt['sources'])} sources]\n")
        elif evt['type'] == 'done':
            print('\n--- done turn_id=', evt['turn_id'])

# Notebooks already run inside an event loop, so await directly instead of asyncio.run().
await run()

INFO: HTTP Request: POST https://azr-oai-dai-sandbox4-001.openai.azure.com/openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-08-01-preview "HTTP/1.1 200 OK"
INFO: Request URL: 'https://agent-ai-search5yc2.search.windows.net/indexes('pdg-was-multimodal-rag-2')/docs/search.post.search?api-version=2024-05-01-preview'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '34598'
    'api-key': 'REDACTED'
    'Accept': 'application/json;odata.metadata=none'
    'x-ms-client-request-id': '6b90d8c5-4882-11f1-8de8-df1619e59047'
    'User-Agent': 'azsdk-python-search-documents/11.6.0b4 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
A body is sent with the request
INFO: Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; odata.metadata=none; odata.streaming=true; charset=utf-8'
    'Content-Encoding': 'REDACTED'
    'Vary': 'REDACTED'
    'Strict-Transport-Security': 'REDAC

[5 sources]



INFO: HTTP Request: POST https://azr-oai-dai-sandbox4-001.openai.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-08-01-preview "HTTP/1.1 200 OK"


The document contains the following images:

1. **Page 15**: A flowchart detailing a process for handling a user query through various nodes and tasks, starting with the "User Query" and including components like the "Planner Node" and "Research Intent Classification" (page 15).

2. **Page 16**: Figure 1, titled "Multi-Agent Research System - Architecture Flow Diagram" (page 16).
--- done turn_id= 77d02ff9-1aa9-47c2-b860-e25e5baa10bf
